# Dynamic and Hybrid Conditioning for Compositional Image Retrieval

**Deep Learning 2026 — Project notebook**

*Team: <YOUR NAMES HERE>*

---

## Abstract

Given a reference face image $v_{ref}$ and a set of textual attribute constraints
(positive, e.g. `+Eyeglasses`, or negative, e.g. `-Smiling`), the task is to retrieve,
from the CelebA test split (19,962 images), the images that **preserve the identity of the
reference** while **satisfying every constraint**.

We build three methods of increasing sophistication and compare them on the official
benchmark (`celeba_evaluation.json`, Recall@K / Precision@K at $K \in \{1, 5, 10\}$):

1. **Level 1 — Zero-shot latent arithmetic** (mandatory baseline): $\hat q = \mathrm{norm}(v_{ref} + \sum_i s_i\, t_i)$.
2. **Level 2 — Visual attribute directions** (training-free): replaces cross-modal text
   arithmetic with intra-modal attribute directions estimated from the CelebA *train* split,
   optionally composed on the unit sphere via logarithmic/exponential maps.
3. **Level 3 — Gated cross-attention fusion** (training-based, our main contribution):
   a lightweight learned module (~4M params, frozen CLIP) that *dynamically* weighs the
   conditions with cross-attention and controls the modification strength with a learned
   gate, trained with InfoNCE on triplets mined from the train split with the same
   relaxed-Hamming rule used to build the benchmark.

---

## How to run / debugging checklist

The notebook is designed to be debugged **one section at a time, top to bottom**.
Every expensive artifact is cached on Google Drive, so re-running is cheap.

| Section | What to verify before moving on | Expected cost (Colab T4) |
|---|---|---|
| §1–2 Setup + data | `len(test) == 19962`, `len(train) == 162770`, 40 attributes | ~2 min (unzip) |
| §3 Embeddings | shapes `(19962, 512)` / `(162770, 512)`, rows unit-norm | ~5 min + ~35 min, **once** (then cached) |
| §4 Benchmark | all 14 queries parse, harness self-test passes | seconds |
| §5 Level 1 | macro R@10 well above random chance (~1.3%) | ~1 min |
| §6 Level 2 | beats Level 1 on macro R@10 | ~1 min |
| §7 Level 3 | training loss decreases, synthetic val R@10 improves, then beats Level 2 on the benchmark | ~5 min mining + ~10 min training, cached |
| §8 Results | comparative tables + qualitative examples | ~2 min |

> **Tip:** set `SMOKE_TEST = True` in the config cell for a first fast end-to-end run
> (fewer benchmark sources, fewer training examples, 2 epochs). Set it back to `False`
> for the real numbers. Smoke artifacts are cached under different filenames.


## 1. Environment setup

The notebook runs on Google Colab (GPU runtime recommended). It expects on Google Drive,
inside `MyDrive/datasets/`:

- `celeba.zip` — the CelebA dataset (provided by the course),
- `celeba_evaluation.json` — the official benchmark file.

All caches (CLIP embeddings, mined training examples, model checkpoint) are written to the
same Drive folder so they survive runtime disconnections. The notebook also runs locally
for debugging: it falls back to the repo's `database/` folder and skips the Colab-specific
steps.

In [ ]:
%pip install -q transformers accelerate

In [ ]:
import json
import math
import random
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

IN_COLAB = "google.colab" in sys.modules

# --- Paths -----------------------------------------------------------------
if IN_COLAB:
    DRIVE_DIR = Path("/content/drive/MyDrive/datasets")  # zip, benchmark JSON, caches
    DATA_ROOT = Path("/content/datasets")                # fast local SSD (wiped on disconnect)
else:
    DRIVE_DIR = Path("database")                         # local fallback: repo folder
    DATA_ROOT = Path("datasets")

EVAL_JSON_PATH = DRIVE_DIR / "celeba_evaluation.json"
MODEL_NAME = "openai/clip-vit-base-patch32"              # mandatory model for the assignment

# --- Global switches ---------------------------------------------------------
SEED = 42
K_VALUES = (1, 5, 10)     # evaluation cutoffs required by the assignment
SMOKE_TEST = False        # True = fast reduced run to debug the whole pipeline
SMOKE_SUFFIX = "_smoke" if SMOKE_TEST else ""

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if device.type == "cuda":
    torch.cuda.manual_seed_all(SEED)

# Fixed categorical colors for all charts (validated colorblind-safe palette)
METHOD_COLORS = {
    "L1 zero-shot arithmetic": "#2a78d6",   # blue
    "L2 visual directions": "#1baf7a",      # aqua
    "L3 gated cross-attention": "#eda100",  # yellow
}

print(f"Colab: {IN_COLAB} | device: {device} | smoke test: {SMOKE_TEST}")

In [ ]:
# Colab-only: mount Drive and unzip CelebA to the runtime's local SSD.
# The zip stays on Drive; the extracted copy is recreated at every new runtime.
if IN_COLAB:
    import subprocess

    from google.colab import drive

    drive.mount("/content/drive")
    DATA_ROOT.mkdir(parents=True, exist_ok=True)
    if not (DATA_ROOT / "celeba").exists():
        print("Unzipping CelebA to local SSD (1-2 minutes)...")
        subprocess.run(
            ["unzip", "-q", str(DRIVE_DIR / "celeba.zip"), "-d", str(DATA_ROOT)],
            check=True,
        )
    print("Dataset ready.")

## 2. Data loading

We load two splits of CelebA through `torchvision.datasets.CelebA`:

- **test** (19,962 images): the retrieval corpus. All benchmark sources and targets are
  *dataset indices* into this split (**not** filenames — see the assignment's warning).
- **train** (162,770 images): never touched by the evaluation. We use it in §6 to estimate
  visual attribute directions and in §7 to mine training triplets for the fusion module.

Each image comes with 40 binary attribute annotations (`celeba.attr`, values in $\{0,1\}$).

In [ ]:
from torchvision.datasets import CelebA

celeba_test = CelebA(root=DATA_ROOT, split="test", download=False)
celeba_train = CelebA(root=DATA_ROOT, split="train", download=False)

# Some torchvision versions append a trailing empty name — filter it out.
attr_names = [n for n in celeba_test.attr_names if n]
N_ATTRS = len(attr_names)
idx2attribute = dict(enumerate(attr_names))
attribute2idx = {name: idx for idx, name in idx2attribute.items()}

# Binary attribute matrices, shape (N, 40), values in {0, 1}
attrs_test = celeba_test.attr.to(torch.uint8)
attrs_train = celeba_train.attr.to(torch.uint8)
attrs_train_dev = attrs_train.to(device)  # small (~6.5 MB), used a lot in §6-7

assert len(celeba_test) == 19962, f"Unexpected test split size: {len(celeba_test)}"
assert N_ATTRS == 40, f"Expected 40 attributes, got {N_ATTRS}"
assert attrs_test.shape == (len(celeba_test), 40)
assert set(attrs_train.unique().tolist()) <= {0, 1}, "Attributes must be in {0,1}"

print(f"test: {len(celeba_test)} images | train: {len(celeba_train)} images")
print(f"attributes ({N_ATTRS}): {attr_names[:6]} ...")

## 3. Offline feature extraction

Following the CLAY methodology, the visual database is built **once, offline, and kept
frozen**: we embed every image with CLIP ViT-B/32, L2-normalize (CLIP similarity is cosine
similarity, so all vectors live on the unit hypersphere $\mathbb{S}^{511}$), and cache the
result on Drive. All three methods then operate purely on these frozen 512-d embeddings —
no CLIP forward pass is ever needed again, which is what makes training in §7 take minutes.

Expected one-time cost on a T4: ~5 min (test split) + ~35 min (train split).

In [ ]:
from transformers import CLIPModel, CLIPProcessor

processor = CLIPProcessor.from_pretrained(MODEL_NAME)
model = CLIPModel.from_pretrained(MODEL_NAME).to(device)
model.eval()
for p in model.parameters():
    p.requires_grad_(False)  # CLIP stays frozen for the whole project

print(f"Model loaded: {MODEL_NAME}")

In [ ]:
class ImageOnlyDataset(Dataset):
    # Wraps a CelebA split and yields only the PIL image;
    # CLIPProcessor handles resize / center-crop / normalization.
    def __init__(self, base):
        self.base = base

    def __len__(self):
        return len(self.base)

    def __getitem__(self, idx):
        return self.base[idx][0]


def clip_collate(images):
    return processor(images=images, return_tensors="pt")


@torch.no_grad()
def extract_image_embeddings(split_ds, cache_path, batch_size=256):
    # Returns (N, 512) L2-normalized CLIP embeddings, cached on Drive.
    cache_path = Path(cache_path)
    if cache_path.exists():
        # weights_only=False: this is our own trusted cache file.
        blob = torch.load(cache_path, map_location="cpu", weights_only=False)
        assert blob["model"] == MODEL_NAME, "Cache was built with a different model"
        emb = blob["embeddings"]
        print(f"Loaded cached embeddings {tuple(emb.shape)} from {cache_path}")
        return emb

    loader = DataLoader(
        ImageOnlyDataset(split_ds),
        batch_size=batch_size,
        shuffle=False,   # keep original order so row i == dataset index i
        num_workers=2,
        collate_fn=clip_collate,
    )
    chunks = []
    for batch in tqdm(loader, desc=f"Embedding -> {cache_path.name}"):
        # get_image_features = vision transformer + projection to the shared 512-d space
        feats = model.get_image_features(pixel_values=batch["pixel_values"].to(device))
        chunks.append(F.normalize(feats, dim=-1).cpu())
    emb = torch.cat(chunks)
    torch.save({"embeddings": emb, "model": MODEL_NAME}, cache_path)
    print(f"Saved {tuple(emb.shape)} to {cache_path}")
    return emb

In [ ]:
test_emb = extract_image_embeddings(celeba_test, DRIVE_DIR / "celeba_clip_test_emb.pt")
train_emb = extract_image_embeddings(celeba_train, DRIVE_DIR / "celeba_clip_train_emb.pt")

# Device copies used for all similarity computations
test_emb_dev = test_emb.to(device)
train_emb_dev = train_emb.to(device)

# Sanity: shapes and unit norms
assert test_emb.shape == (len(celeba_test), 512)
assert train_emb.shape == (len(celeba_train), 512)
assert torch.allclose(test_emb.norm(dim=-1), torch.ones(len(test_emb)), atol=1e-3)
print("Embeddings OK.")

## 4. Benchmark: queries, parsing, evaluation harness

`celeba_evaluation.json` is a list of 14 query entries (the query `-Young` appears twice —
we simply evaluate every entry as given). Each entry contains:

- `query`: e.g. `"+Eyeglasses"` or `"-Smiling, +Eyeglasses, +Wearing_Hat"` — attribute
  names match `celeba.attr_names` exactly, comma-separated, prefixed by `+`/`-`;
- `ground_truth`: a dict mapping **source indices (as strings)** to the list of valid
  target indices. Both are *PyTorch dataset indices* into the test split.

A target is valid iff it strictly satisfies the query constraints **and** its remaining
attributes are within Hamming distance ≤ 2 of the source (identity preservation). Only
sources with ≥ 5 valid targets are included.

**Metrics.** For each query we average over its sources: Recall@K (1 if at least one valid
target is in the top-K, the primary metric) and Precision@K, at $K \in \{1,5,10\}$.
For reference, a random ranker with ~26 valid targets in a corpus of 19,962 gets
$R@10 \approx 1 - (1 - 26/19962)^{10} \approx 1.3\%$.

All methods are evaluated by the same harness: they only differ in a
`make_queries(conds, source_indices) -> (S, 512)` function that produces the composite
query embeddings; ranking is always cosine similarity against the frozen test corpus,
with the source image itself excluded from the ranking.

In [ ]:
with open(EVAL_JSON_PATH, "r") as f:
    annotations = json.load(f)


def parse_query(query_str):
    # "-Smiling, +Eyeglasses" -> [("Smiling", -1), ("Eyeglasses", +1)]
    conds = []
    for part in query_str.split(","):
        part = part.strip()
        sign = +1 if part[0] == "+" else -1
        attr = part[1:].strip()
        assert attr in attribute2idx, f"Unknown attribute in query: {attr!r}"
        conds.append((attr, sign))
    return conds


# Sanity: every benchmark query must parse against the CelebA attribute names
for entry in annotations:
    parse_query(entry["query"])

print(f"{len(annotations)} benchmark queries, all parsed OK:")
for entry in annotations:
    n_src = len(entry["ground_truth"])
    avg_tgt = np.mean([len(v) for v in entry["ground_truth"].values()])
    print(f"  {entry['query']:<45} sources: {n_src:>5} | avg targets: {avg_tgt:5.1f}")

In [ ]:
def evaluate_retrieval(retrieved_indices, ground_truth_indices, k):
    # Metric implementation provided in the course skeleton (kept verbatim in spirit):
    # Recall@K is a per-source hit indicator, Precision@K the fraction of hits in top-K.
    top_k_retrieved = retrieved_indices[:k]
    hits = set(top_k_retrieved).intersection(set(ground_truth_indices))
    num_hits = len(hits)
    return {
        f"Recall@{k}": 1 if num_hits > 0 else 0,
        f"Precision@{k}": num_hits / k,
    }


@torch.no_grad()
def run_benchmark(annotations, make_queries, method_name,
                  db_emb=None, k_values=K_VALUES, chunk_size=2048):
    # Evaluates one method on every benchmark entry.
    # make_queries(conds, src_idx_tensor) must return (S, 512) unit-norm query
    # embeddings on `device`, one per source image.
    if db_emb is None:
        db_emb = test_emb_dev
    max_k = max(k_values)
    rows = []
    for entry in tqdm(annotations, desc=method_name):
        conds = parse_query(entry["query"])
        gt = entry["ground_truth"]
        src = torch.tensor([int(s) for s in gt.keys()], dtype=torch.long)
        if SMOKE_TEST:
            src = src[:200]  # reduced run for fast end-to-end debugging

        tops = []
        for i in range(0, len(src), chunk_size):
            chunk = src[i:i + chunk_size]
            Q = make_queries(conds, chunk)                      # (S, 512)
            sims = Q @ db_emb.T                                 # (S, N_corpus)
            # Never retrieve the source image itself
            sims[torch.arange(len(chunk), device=device), chunk.to(device)] = -float("inf")
            tops.append(sims.topk(max_k, dim=1).indices.cpu())
        top = torch.cat(tops)

        row = {"query": entry["query"], "sources": len(src)}
        for k in k_values:
            recalls, precisions = [], []
            for i, s in enumerate(src.tolist()):
                m = evaluate_retrieval(top[i].tolist(), gt[str(s)], k)
                recalls.append(m[f"Recall@{k}"])
                precisions.append(m[f"Precision@{k}"])
            row[f"R@{k}"] = float(np.mean(recalls))
            row[f"P@{k}"] = float(np.mean(precisions))
        rows.append(row)

    df = pd.DataFrame(rows)
    macro = {"query": "MACRO AVERAGE", "sources": int(df["sources"].sum())}
    for c in df.columns[2:]:
        macro[c] = df[c].mean()
    df.loc[len(df)] = macro
    df.insert(0, "method", method_name)
    return df


# Harness self-test (same cases as the course skeleton)
assert evaluate_retrieval([], [3, 2, 1], 1) == {"Recall@1": 0, "Precision@1": 0.0}
assert evaluate_retrieval([3, 9, 9], [3, 2, 1], 1) == {"Recall@1": 1, "Precision@1": 1.0}
assert evaluate_retrieval([9, 9, 3], [3, 2, 1], 3)["Recall@3"] == 1
print("Harness self-test OK.")

## 5. Level 1 — Zero-shot latent arithmetic (mandatory baseline)

**Method.** Each attribute $a$ is described by a natural-language prompt (prompt
engineering matters for CLIP, so we use hand-written phrases like *"a photo of a person
wearing eyeglasses"* rather than raw attribute names). With $t_a$ the normalized CLIP text
embedding of the prompt and $s_i \in \{+1, -1\}$ the sign of each condition, the composite
query is plain latent arithmetic:

$$\hat q \;=\; \frac{v_{ref} + \sum_{i} s_i\, t_{a_i}}{\left\lVert v_{ref} + \sum_{i} s_i\, t_{a_i} \right\rVert}$$

**Known limitations** (which motivate Levels 2 and 3):

- *Fixed weights*: every condition contributes with weight 1, regardless of whether the
  reference already satisfies it or of conflicts between conditions.
- *Modality gap*: CLIP image and text embeddings occupy two disjoint cones on the sphere,
  so adding a text vector to an image vector moves the query partly along the (semantically
  meaningless) image→text direction.
- *Negation*: CLIP behaves like a bag-of-words model cross-modally; subtracting the
  embedding of "smiling" is a crude proxy for "not smiling".

In [ ]:
# Natural-language prompts. Hand-written phrases for the attributes used by the
# benchmark; a generic template covers the remaining ones.
PROMPT_OVERRIDES = {
    "Smiling": "a photo of a smiling person",
    "Eyeglasses": "a photo of a person wearing eyeglasses",
    "Heavy_Makeup": "a photo of a person wearing heavy makeup",
    "Male": "a photo of a man",
    "Young": "a photo of a young person",
    "Blond_Hair": "a photo of a person with blond hair",
    "Black_Hair": "a photo of a person with black hair",
    "Wavy_Hair": "a photo of a person with wavy hair",
    "Mustache": "a photo of a person with a mustache",
    "Chubby": "a photo of a chubby person",
    "Wearing_Hat": "a photo of a person wearing a hat",
    "Wearing_Lipstick": "a photo of a person wearing lipstick",
    "No_Beard": "a photo of a clean-shaven person",
    "5_o_Clock_Shadow": "a photo of a person with stubble",
}


def build_prompt(attr_name):
    generic = f"a photo of a person with {attr_name.replace('_', ' ').lower()}"
    return PROMPT_OVERRIDES.get(attr_name, generic)


@torch.no_grad()
def encode_texts(prompts, batch_size=64):
    feats = []
    for i in range(0, len(prompts), batch_size):
        inputs = processor(
            text=prompts[i:i + batch_size], return_tensors="pt", padding=True
        ).to(device)
        feats.append(model.get_text_features(**inputs))
    return F.normalize(torch.cat(feats), dim=-1)


# (40, 512) matrix: one normalized text embedding per attribute, row = attribute index.
# Reused by Level 1 (arithmetic) and Level 3 (condition tokens).
text_feats = encode_texts([build_prompt(idx2attribute[i]) for i in range(N_ATTRS)])
print("text_feats:", tuple(text_feats.shape))
print("example prompt:", build_prompt("Wearing_Lipstick"))

In [ ]:
def make_queries_baseline(conds, src_idx):
    # q = normalize(v_ref + sum_i sign_i * t_i)  — fixed unit weights.
    t = torch.stack(
        [sign * text_feats[attribute2idx[attr]] for attr, sign in conds]
    ).sum(dim=0)
    v = test_emb_dev[src_idx.to(device)]
    return F.normalize(v + t, dim=-1)


df_baseline = run_benchmark(annotations, make_queries_baseline, "L1 zero-shot arithmetic")
df_baseline.round(3)

## 6. Level 2 — Visual attribute directions (training-free)

**Idea.** The baseline's weakest assumption is that a *text* embedding is a good direction
to move an *image* embedding. Because of the modality gap it is not. But the CelebA train
split gives us attribute labels for 162,770 images, so we can estimate the direction of
each attribute *inside the image cone*, where the arithmetic actually happens:

$$\delta_a \;=\; \mathbb{E}\big[v \mid a{=}1\big] \;-\; \mathbb{E}\big[v \mid a{=}0\big]$$

This is fully training-free (two means per attribute) and intra-modal — no gap to cross.

**Spherical (geodesic) variant.** CLIP embeddings live on the unit sphere, so Euclidean
averaging and addition are geometrically sloppy (Berasi et al., 2025). We therefore also
compute the directions in the tangent space at the intrinsic mean $\mu$ of the image cone,
using the sphere's logarithmic and exponential maps:

$$\mathrm{Log}_{\mu}(x) = \frac{\theta}{\sin\theta}\,\big(x - \cos\theta \,\mu\big),
\qquad \theta = \arccos\langle \mu, x\rangle$$

$$\mathrm{Exp}_{\mu}(u) = \cos(\lVert u\rVert)\,\mu + \sin(\lVert u\rVert)\,\frac{u}{\lVert u\rVert}$$

Directions are tangent-space class means, and the query is composed on the manifold:

$$\hat q \;=\; \mathrm{Exp}_{\mu}\!\Big(\mathrm{Log}_{\mu}(v_{ref}) + \gamma \sum_i s_i\, \delta_{a_i}\Big)$$

We report both the linear and the geodesic variant. The step size $\gamma$ is kept at 1.0;
it could be tuned, but only on synthetic train-split queries (never on the benchmark — that
would be test leakage). The mined validation split built in §7 is the right place for such
tuning if desired.

In [ ]:
EPS = 1e-7


def log_map(x, mu):
    # Sphere log map: maps unit vectors x to the tangent space at mu.
    cos_t = (x @ mu).clamp(-1 + EPS, 1 - EPS)
    theta = torch.acos(cos_t)
    u = x - cos_t.unsqueeze(-1) * mu
    u = u / u.norm(dim=-1, keepdim=True).clamp_min(EPS)
    return theta.unsqueeze(-1) * u


def exp_map(u, mu):
    # Sphere exp map: maps tangent vectors at mu back to the sphere.
    n = u.norm(dim=-1, keepdim=True).clamp_min(EPS)
    return torch.cos(n) * mu + torch.sin(n) * (u / n)


with torch.no_grad():
    # Intrinsic mean of the image cone (normalized Euclidean mean is an excellent
    # approximation of the Frechet mean for a tight cluster like CLIP's cone).
    mu = F.normalize(train_emb_dev.mean(dim=0), dim=0)

    attr_bool = attrs_train_dev.bool()                     # (N_train, 40)
    tan_train = log_map(train_emb_dev, mu)                 # (N_train, 512)

    dirs_lin, dirs_tan = [], []
    for a in range(N_ATTRS):
        pos_mask, neg_mask = attr_bool[:, a], ~attr_bool[:, a]
        dirs_lin.append(train_emb_dev[pos_mask].mean(0) - train_emb_dev[neg_mask].mean(0))
        dirs_tan.append(tan_train[pos_mask].mean(0) - tan_train[neg_mask].mean(0))
    dirs_lin = torch.stack(dirs_lin)                       # (40, 512)
    dirs_tan = torch.stack(dirs_tan)                       # (40, 512)
    del tan_train  # free ~330 MB

print("direction norms (tangent):",
      {idx2attribute[a]: round(dirs_tan[a].norm().item(), 3)
       for a in [attribute2idx["Smiling"], attribute2idx["Male"],
                 attribute2idx["Eyeglasses"]]})

In [ ]:
GAMMA = 1.0  # step size along the composed direction (see leakage note above)


def make_queries_dirs_geodesic(conds, src_idx):
    v = test_emb_dev[src_idx.to(device)]
    d = torch.stack(
        [sign * dirs_tan[attribute2idx[attr]] for attr, sign in conds]
    ).sum(dim=0)
    q = exp_map(log_map(v, mu) + GAMMA * d, mu)
    return F.normalize(q, dim=-1)  # exp map is unit-norm already; normalize for safety


def make_queries_dirs_linear(conds, src_idx):
    v = test_emb_dev[src_idx.to(device)]
    d = torch.stack(
        [sign * dirs_lin[attribute2idx[attr]] for attr, sign in conds]
    ).sum(dim=0)
    return F.normalize(v + GAMMA * d, dim=-1)


df_dirs = run_benchmark(annotations, make_queries_dirs_geodesic, "L2 visual directions")
df_dirs_lin = run_benchmark(
    annotations, make_queries_dirs_linear, "L2 visual directions (linear ablation)"
)

print("\nGeodesic variant:")
display(df_dirs.round(3))
print("Linear ablation:")
display(df_dirs_lin.round(3))

## 7. Level 3 — Gated cross-attention fusion (main contribution)

The training-free methods still apply **static** directions: the same $\delta_a$ with the
same weight for every reference image, and no interaction between conditions. Our fusion
module $\Phi$ makes both dynamic, while CLIP stays frozen and retrieval remains a single
cosine similarity against the fixed database (CLAY-style efficiency).

### 7.1 Architecture

Inputs: $v_{ref} \in \mathbb{R}^{512}$ (frozen image embedding), condition text embeddings
$t_1,\dots,t_k$ with signs $s_1,\dots,s_k$.

**Condition tokens.** Each condition becomes a token carrying content *and* direction:
$$c_i = W_c\, t_i + e_{s_i}, \qquad e_{+}, e_{-} \in \mathbb{R}^{512} \text{ learned}$$
The learned sign embeddings let the model *learn* what "remove attribute" means instead of
assuming it is the Euclidean opposite of "add attribute" (CLIP handles negation poorly).

**Condition self-attention (1 layer).** The tokens attend to each other so that
interacting/conflicting conditions (e.g. `+Wearing_Lipstick, -Heavy_Makeup`) can negotiate
before the image looks at them.

**Cross-attention (image queries the conditions).**
$$\alpha = \mathrm{softmax}\!\Big(\tfrac{(W_q v_{ref})(W_k C)^\top}{\sqrt{d}}\Big), \qquad
\Delta = W_o\,(\alpha\, W_v C)$$
The weights $\alpha$ depend on the reference: for the same query, a reference that already
almost satisfies one condition shifts attention to the others. $W_o$ is **zero-initialized**,
so at initialization $\Phi(v_{ref}, \cdot) = v_{ref}$ exactly: training starts from the
identity and only learns the *correction*.

**Gate (how much to move).**
$$g = \sigma\big(\mathrm{MLP}([\,v_{ref}\,;\,\Delta\,])\big) \in (0,1)^{512}, \qquad
\hat q = \frac{v_{ref} + g \odot \Delta}{\lVert v_{ref} + g \odot \Delta \rVert}$$
The residual form structurally guarantees the query never forgets the reference identity —
which is half of the task (targets must stay within Hamming ≤ 2 of the source).

### 7.2 Training data (mined, no manual labels)

The benchmark ground truth follows a *declared* rule, so we replicate it on the **train
split** to mine supervision that is exactly aligned with the evaluation, with zero leakage
(different split, different images):

1. sample a source image and $k \in \{1,2,3\}$ attributes (biased towards benchmark
   attributes); the sign of each condition is determined by flipping the source's value,
   so the source never satisfies its own query — exactly like the benchmark;
2. **positives** = train images that satisfy all flipped values and have Hamming ≤ 2 on the
   remaining attributes; examples with < 5 positives are rejected (same filter as the benchmark);
3. **hard negatives** of the two failure modes:
   *identity-negatives* (Hamming ≤ 2 but ≥ 1 constraint violated → punishes ignoring the
   text) and *constraint-negatives* (constraints satisfied but Hamming ≥ 6 → punishes
   ignoring the reference).

Validation examples are mined from a **disjoint pool of source images** and used for early
stopping and model selection (synthetic Recall@10 on the train corpus).

### 7.3 Loss

InfoNCE with temperature $\tau = 0.07$ over the positive, the mined hard negatives, and
the other in-batch positives as easy negatives:

$$\mathcal{L} = -\log
\frac{e^{\langle \hat q, v^+\rangle/\tau}}
{e^{\langle \hat q, v^+\rangle/\tau}
 + \sum_{v^- \in \mathrm{hard}} e^{\langle \hat q, v^-\rangle/\tau}
 + \sum_{j \neq i} e^{\langle \hat q, v_j^+\rangle/\tau}}$$

### 7.4 Hyperparameters

| | |
|---|---|
| Module size | 512-d, 4 heads, 1 self-attn + 1 cross-attn layer, ~4M params |
| Optimizer | AdamW, lr $10^{-4}$, weight decay $10^{-2}$ |
| Batch / epochs | 256, up to 15 epochs, early stopping (patience 3) on val R@10 |
| Hard negatives | 8 per example (4 identity + 4 constraint) |
| Training examples | 30,000 mined (+2,000 validation) |

In [ ]:
# ---- 7.2 Training-data mining ----------------------------------------------
# Attributes that appear in the benchmark queries (used to bias sampling)
BENCH_ATTR_IDX = sorted(
    {attribute2idx[a] for e in annotations for a, _ in parse_query(e["query"])}
)

MIN_POSITIVES = 5   # same filter as the official benchmark
HAM_ID = 2          # identity budget (relaxed Hamming), same as the benchmark
HAM_FAR = 6         # "clearly different identity" threshold for constraint-negatives
CAP_POS, CAP_NEG = 256, 64

N_TRAIN_EX = 2000 if SMOKE_TEST else 30000
N_VAL_EX = 500 if SMOKE_TEST else 2000


def mine_example(src, cond_idx):
    # cond_idx: LongTensor of attribute indices to flip on the source.
    # Returns None if the query has fewer than MIN_POSITIVES valid targets.
    src_vals = attrs_train_dev[src, cond_idx]
    tgt_vals = 1 - src_vals                        # flip -> source never satisfies the query
    signs = torch.where(tgt_vals == 1, 1, -1)      # +1 = add attribute, -1 = remove

    sat = (attrs_train_dev[:, cond_idx] == tgt_vals).all(dim=1)
    other = torch.ones(N_ATTRS, dtype=torch.bool, device=device)
    other[cond_idx] = False
    ham = (attrs_train_dev[:, other] != attrs_train_dev[src, other]).sum(dim=1)

    pos = sat & (ham <= HAM_ID)
    pos[src] = False
    if int(pos.sum()) < MIN_POSITIVES:
        return None

    id_neg = (~sat) & (ham <= HAM_ID)   # right identity, wrong constraints
    id_neg[src] = False
    con_neg = sat & (ham >= HAM_FAR)    # right constraints, wrong identity

    def pick(mask, cap):
        idx = mask.nonzero(as_tuple=True)[0]
        if len(idx) > cap:
            idx = idx[torch.randperm(len(idx), device=device)[:cap]]
        return idx.cpu().numpy().astype(np.int32)

    return {
        "source": int(src),
        "cond_idx": cond_idx.cpu().numpy().astype(np.int16),
        "signs": signs.cpu().numpy().astype(np.int8),
        "positives": pick(pos, CAP_POS),
        "id_negs": pick(id_neg, CAP_NEG),
        "con_negs": pick(con_neg, CAP_NEG),
    }


def generate_examples(n_examples, source_pool, rng,
                      p_bench=0.7, k_probs=(0.5, 0.35, 0.15)):
    out = []
    attempts, max_attempts = 0, n_examples * 30
    pbar = tqdm(total=n_examples, desc="Mining examples")
    while len(out) < n_examples and attempts < max_attempts:
        attempts += 1
        src = int(rng.choice(source_pool))
        k = int(rng.choice([1, 2, 3], p=list(k_probs)))
        pool = BENCH_ATTR_IDX if rng.random() < p_bench else list(range(N_ATTRS))
        if k > len(pool):
            continue
        cond_idx = torch.tensor(
            sorted(rng.choice(pool, size=k, replace=False).tolist()),
            dtype=torch.long, device=device,
        )
        ex = mine_example(src, cond_idx)
        if ex is not None:
            out.append(ex)
            pbar.update(1)
    pbar.close()
    if len(out) < n_examples:
        print(f"WARNING: mined only {len(out)}/{n_examples} examples "
              f"(hit the attempt cap) — consider relaxing the filters.")
    return out


EXAMPLES_PATH = DRIVE_DIR / f"fusion_examples{SMOKE_SUFFIX}.pt"
if EXAMPLES_PATH.exists():
    blob = torch.load(EXAMPLES_PATH, map_location="cpu", weights_only=False)
    train_examples, val_examples = blob["train"], blob["val"]
    print(f"Loaded {len(train_examples)} train / {len(val_examples)} val mined examples.")
else:
    rng = np.random.default_rng(SEED)
    perm = rng.permutation(len(celeba_train))
    val_pool, train_pool = perm[:8000], perm[8000:]  # disjoint source pools
    train_examples = generate_examples(N_TRAIN_EX, train_pool, rng)
    val_examples = generate_examples(N_VAL_EX, val_pool, rng)
    torch.save({"train": train_examples, "val": val_examples}, EXAMPLES_PATH)
    print(f"Saved mined examples to {EXAMPLES_PATH}")

avg_pos = np.mean([len(e["positives"]) for e in train_examples])
avg_k = np.mean([len(e["cond_idx"]) for e in train_examples])
print(f"avg positives/example: {avg_pos:.1f} | avg conditions/example: {avg_k:.2f}")

In [ ]:
# ---- Dataset / collate -------------------------------------------------------
text_feats_cpu = text_feats.cpu()


class FusionDataset(Dataset):
    # Each item: frozen reference embedding, condition tokens, one random positive,
    # and n_hard mined hard negatives (half identity-, half constraint-negatives).
    def __init__(self, examples, img_emb, n_hard=8):
        self.examples = examples
        self.img_emb = img_emb  # (N_train, 512) on CPU
        self.n_hard = n_hard

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, i):
        ex = self.examples[i]
        pos = int(np.random.choice(ex["positives"]))

        negs = []
        half = self.n_hard // 2
        for pool in (ex["id_negs"], ex["con_negs"]):
            if len(pool) > 0:
                negs += np.random.choice(pool, size=half,
                                         replace=len(pool) < half).tolist()
        while len(negs) < self.n_hard:  # fallback: random corpus negatives
            negs.append(np.random.randint(len(self.img_emb)))

        cond_idx = torch.from_numpy(ex["cond_idx"].astype(np.int64))
        return {
            "v_ref": self.img_emb[ex["source"]],
            "cond_emb": text_feats_cpu[cond_idx],
            "sign_ids": torch.from_numpy((ex["signs"] < 0).astype(np.int64)),  # 0=+, 1=-
            "v_pos": self.img_emb[pos],
            "v_negs": self.img_emb[torch.tensor(negs, dtype=torch.long)],
        }


def collate_fusion(items):
    # Pads the variable-length condition sequences; pad_mask True = padding.
    B = len(items)
    K = max(len(it["sign_ids"]) for it in items)
    cond_emb = torch.zeros(B, K, 512)
    sign_ids = torch.zeros(B, K, dtype=torch.long)
    pad_mask = torch.ones(B, K, dtype=torch.bool)
    for b, it in enumerate(items):
        k = len(it["sign_ids"])
        cond_emb[b, :k] = it["cond_emb"]
        sign_ids[b, :k] = it["sign_ids"]
        pad_mask[b, :k] = False
    return {
        "v_ref": torch.stack([it["v_ref"] for it in items]),
        "cond_emb": cond_emb,
        "sign_ids": sign_ids,
        "pad_mask": pad_mask,
        "v_pos": torch.stack([it["v_pos"] for it in items]),
        "v_negs": torch.stack([it["v_negs"] for it in items]),
    }


def build_condition_tensors(conds, n_repeat):
    # Broadcasts one fixed condition list [(attr_name, sign), ...] to a batch of
    # n_repeat sources. Used at evaluation time (all sources share the query).
    idx = torch.tensor([attribute2idx[a] for a, _ in conds], device=device)
    cond_emb = text_feats[idx].unsqueeze(0).expand(n_repeat, -1, -1)
    sign_ids = torch.tensor(
        [0 if s > 0 else 1 for _, s in conds], dtype=torch.long, device=device
    ).unsqueeze(0).expand(n_repeat, -1)
    return cond_emb, sign_ids

In [ ]:
# ---- 7.1 Model + 7.3 loss ----------------------------------------------------
class GatedCrossAttentionFusion(nn.Module):
    def __init__(self, dim=512, n_heads=4, gate_hidden=256, use_self_attn=True):
        super().__init__()
        self.cond_proj = nn.Linear(dim, dim)
        self.sign_emb = nn.Embedding(2, dim)  # index 0 = "+", index 1 = "-"
        self.use_self_attn = use_self_attn
        if use_self_attn:
            self.self_attn = nn.TransformerEncoderLayer(
                dim, n_heads, dim_feedforward=2 * dim, dropout=0.1, batch_first=True
            )
        self.cross_attn = nn.MultiheadAttention(dim, n_heads, batch_first=True)
        self.out_proj = nn.Linear(dim, dim)
        # Zero-init: at initialization the module is exactly the identity on v_ref,
        # so training starts from the reference and only learns the correction.
        nn.init.zeros_(self.out_proj.weight)
        nn.init.zeros_(self.out_proj.bias)
        self.gate = nn.Sequential(
            nn.Linear(2 * dim, gate_hidden),
            nn.ReLU(),
            nn.Linear(gate_hidden, dim),
            nn.Sigmoid(),
        )

    def forward(self, v_ref, cond_emb, sign_ids, pad_mask=None, return_attn=False):
        # v_ref: (B, 512) | cond_emb: (B, K, 512) | sign_ids: (B, K) | pad_mask: (B, K)
        c = self.cond_proj(cond_emb) + self.sign_emb(sign_ids)
        if self.use_self_attn:
            c = self.self_attn(c, src_key_padding_mask=pad_mask)
        delta, attn = self.cross_attn(
            v_ref.unsqueeze(1), c, c, key_padding_mask=pad_mask, need_weights=True
        )
        delta = self.out_proj(delta.squeeze(1))
        g = self.gate(torch.cat([v_ref, delta], dim=-1))
        q = F.normalize(v_ref + g * delta, dim=-1)
        if return_attn:
            return q, attn.squeeze(1), g  # attn: (B, K) weights per condition
        return q


def info_nce_loss(q, v_pos, v_negs, temperature=0.07):
    # Positive at column 0; mined hard negatives; other in-batch positives as
    # easy negatives (diagonal masked because it duplicates the positive).
    l_pos = (q * v_pos).sum(dim=-1, keepdim=True)          # (B, 1)
    l_hard = torch.einsum("bd,bhd->bh", q, v_negs)         # (B, H)
    l_batch = q @ v_pos.T                                   # (B, B)
    l_batch.fill_diagonal_(-float("inf"))
    logits = torch.cat([l_pos, l_hard, l_batch], dim=1) / temperature
    labels = torch.zeros(len(q), dtype=torch.long, device=q.device)
    return F.cross_entropy(logits, labels)


fusion = GatedCrossAttentionFusion().to(device)
n_params = sum(p.numel() for p in fusion.parameters())
print(f"Fusion module parameters: {n_params / 1e6:.2f}M (CLIP stays frozen)")

In [ ]:
# ---- Synthetic validation metric ----------------------------------------------
@torch.no_grad()
def synthetic_recall_at_k(net, examples, corpus_emb, k=10, batch=512):
    # Recall@k of the fusion module on mined examples, retrieving from the train
    # corpus. Used only for early stopping / model selection (never the benchmark).
    net.eval()
    hits = []
    for i in range(0, len(examples), batch):
        chunk = examples[i:i + batch]
        B = len(chunk)
        K = max(len(e["cond_idx"]) for e in chunk)
        cond_emb = torch.zeros(B, K, 512, device=device)
        sign_ids = torch.zeros(B, K, dtype=torch.long, device=device)
        pad_mask = torch.ones(B, K, dtype=torch.bool, device=device)
        src = torch.tensor([e["source"] for e in chunk], dtype=torch.long)
        for b, e in enumerate(chunk):
            kk = len(e["cond_idx"])
            cond_emb[b, :kk] = text_feats[torch.from_numpy(e["cond_idx"].astype(np.int64))]
            sign_ids[b, :kk] = torch.from_numpy((e["signs"] < 0).astype(np.int64))
            pad_mask[b, :kk] = False
        q = net(corpus_emb[src.to(device)], cond_emb, sign_ids, pad_mask)
        sims = q @ corpus_emb.T
        sims[torch.arange(B, device=device), src.to(device)] = -float("inf")
        top = sims.topk(k, dim=1).indices.cpu()
        for b, e in enumerate(chunk):
            hits.append(int(len(set(top[b].tolist()) & set(e["positives"].tolist())) > 0))
    return float(np.mean(hits))


# ---- Training loop -------------------------------------------------------------
CKPT_PATH = DRIVE_DIR / f"fusion_model{SMOKE_SUFFIX}.pt"
RETRAIN = False          # set True to force retraining even if a checkpoint exists
EPOCHS = 2 if SMOKE_TEST else 15
PATIENCE = 3
TAU = 0.07

if CKPT_PATH.exists() and not RETRAIN:
    ckpt = torch.load(CKPT_PATH, map_location=device, weights_only=False)
    fusion.load_state_dict(ckpt["state_dict"])
    history = ckpt["history"]
    print(f"Loaded checkpoint (val R@10 = {ckpt['val_r10']:.3f}) from {CKPT_PATH}")
else:
    loader = DataLoader(
        FusionDataset(train_examples, train_emb),
        batch_size=256, shuffle=True, drop_last=True,
        num_workers=0,  # tensors are already in RAM; workers only add overhead
        collate_fn=collate_fusion,
    )
    opt = torch.optim.AdamW(fusion.parameters(), lr=1e-4, weight_decay=1e-2)

    history = {"loss": [], "val_r10": []}
    best_r10, best_state, bad_epochs = -1.0, None, 0

    for epoch in range(1, EPOCHS + 1):
        fusion.train()
        losses = []
        for batch in tqdm(loader, desc=f"Epoch {epoch}/{EPOCHS}", leave=False):
            batch = {k: v.to(device, non_blocking=True) for k, v in batch.items()}
            q = fusion(batch["v_ref"], batch["cond_emb"],
                       batch["sign_ids"], batch["pad_mask"])
            loss = info_nce_loss(q, batch["v_pos"], batch["v_negs"], TAU)
            opt.zero_grad()
            loss.backward()
            opt.step()
            losses.append(loss.item())

        val_r10 = synthetic_recall_at_k(fusion, val_examples, train_emb_dev)
        history["loss"].append(float(np.mean(losses)))
        history["val_r10"].append(val_r10)
        print(f"epoch {epoch:>2} | loss {history['loss'][-1]:.4f} | val R@10 {val_r10:.3f}")

        if val_r10 > best_r10:
            best_r10, bad_epochs = val_r10, 0
            best_state = {k: v.detach().cpu().clone()
                          for k, v in fusion.state_dict().items()}
        else:
            bad_epochs += 1
            if bad_epochs >= PATIENCE:
                print(f"Early stopping at epoch {epoch}.")
                break

    fusion.load_state_dict(best_state)
    torch.save({"state_dict": best_state, "history": history, "val_r10": best_r10},
               CKPT_PATH)
    print(f"Best val R@10 = {best_r10:.3f} | checkpoint saved to {CKPT_PATH}")

# Learning curves (two panels, one series each — never a dual-axis chart)
if history["loss"]:
    fig, axes = plt.subplots(1, 2, figsize=(10, 3.2))
    epochs_x = range(1, len(history["loss"]) + 1)
    axes[0].plot(epochs_x, history["loss"], color="#2a78d6", linewidth=2)
    axes[0].set_title("Training loss (InfoNCE)")
    axes[1].plot(epochs_x, history["val_r10"], color="#1baf7a", linewidth=2)
    axes[1].set_title("Synthetic validation Recall@10")
    for ax in axes:
        ax.set_xlabel("epoch")
        ax.grid(alpha=0.25)
        ax.spines[["top", "right"]].set_visible(False)
    plt.tight_layout()
    plt.show()

In [ ]:
# ---- Benchmark evaluation of the fusion module --------------------------------
fusion.eval()


def make_queries_fusion(conds, src_idx):
    cond_emb, sign_ids = build_condition_tensors(conds, len(src_idx))
    v = test_emb_dev[src_idx.to(device)]
    return fusion(v, cond_emb, sign_ids, None)  # no padding: same conds for all sources


df_fusion = run_benchmark(annotations, make_queries_fusion, "L3 gated cross-attention")
df_fusion.round(3)

## 8. Results and discussion

> **TODO after running:** fill in the discussion below with the actual numbers — compare
> macro R@K across levels, point out which query families benefit the most (single vs
> composed, additive vs subtractive), and discuss failure cases from the qualitative grids.

In [ ]:
# ---- Comparative tables --------------------------------------------------------
df_all = pd.concat([df_baseline, df_dirs, df_fusion], ignore_index=True)

print("Macro averages:")
display(
    df_all[df_all["query"] == "MACRO AVERAGE"]
    .set_index("method")[["R@1", "R@5", "R@10", "P@1", "P@5", "P@10"]]
    .round(3)
)

# Per-query Recall@10 (pivot_table because '-Young' appears twice in the benchmark)
per_query = df_all[df_all["query"] != "MACRO AVERAGE"].pivot_table(
    index="query", columns="method", values="R@10", aggfunc="mean"
)
print("Per-query Recall@10:")
display(per_query.round(3))

In [ ]:
# ---- Per-query Recall@10 chart --------------------------------------------------
methods = list(METHOD_COLORS)  # fixed order = fixed colors, never cycled
queries = per_query.index.tolist()
y = np.arange(len(queries))
bar_h = 0.26

fig, ax = plt.subplots(figsize=(9, 0.55 * len(queries) + 1.5))
for j, m in enumerate(methods):
    if m not in per_query.columns:
        continue
    vals = per_query[m].values
    bars = ax.barh(y + (j - 1) * bar_h, vals, height=bar_h * 0.92,
                   color=METHOD_COLORS[m], label=m)
    ax.bar_label(bars, fmt="%.2f", padding=2, fontsize=8, color="#52514e")

ax.set_yticks(y, queries)
ax.invert_yaxis()
ax.set_xlabel("Recall@10")
ax.set_xlim(0, min(1.0, per_query.values.max() * 1.25))
ax.grid(axis="x", alpha=0.25)
ax.spines[["top", "right"]].set_visible(False)
ax.legend(loc="lower right", frameon=False)
ax.set_title("Recall@10 per benchmark query")
plt.tight_layout()
plt.show()

In [ ]:
# ---- Qualitative examples: source + top-5 retrieved ------------------------------
@torch.no_grad()
def show_retrieval(entry, make_queries, method_name, n_sources=2, n_top=5, seed=SEED):
    conds = parse_query(entry["query"])
    keys = list(entry["ground_truth"].keys())
    picks = random.Random(seed).sample(keys, min(n_sources, len(keys)))
    for skey in picks:
        s = int(skey)
        Q = make_queries(conds, torch.tensor([s]))
        sims = (Q @ test_emb_dev.T).squeeze(0)
        sims[s] = -float("inf")
        top = sims.topk(n_top).indices.tolist()
        gt_set = set(entry["ground_truth"][skey])

        fig, axes = plt.subplots(1, n_top + 1, figsize=(2.2 * (n_top + 1), 2.8))
        axes[0].imshow(celeba_test[s][0])
        axes[0].set_title(f"source {s}", fontsize=9)
        for j, t in enumerate(top):
            axes[j + 1].imshow(celeba_test[t][0])
            hit = t in gt_set
            axes[j + 1].set_title("HIT" if hit else "miss", fontsize=9,
                                  color="#008300" if hit else "#e34948")
        for ax in axes:
            ax.axis("off")
        fig.suptitle(f"{method_name} | query: {entry['query']}", fontsize=10)
        plt.tight_layout()
        plt.show()


# Success/failure inspection on one simple and one composed query;
# change the indices to explore other queries.
for entry_idx in (1, len(annotations) - 1):
    show_retrieval(annotations[entry_idx], make_queries_baseline, "L1 baseline")
    show_retrieval(annotations[entry_idx], make_queries_fusion, "L3 fusion")

In [ ]:
# ---- Interpretability: attention weights and gate behavior -----------------------
@torch.no_grad()
def attention_report(entry, max_sources=256):
    # Mean cross-attention weight per condition + how far the query moved from
    # the reference (1 - cosine), aggregated over the query's source images.
    conds = parse_query(entry["query"])
    src = torch.tensor([int(s) for s in entry["ground_truth"].keys()][:max_sources])
    cond_emb, sign_ids = build_condition_tensors(conds, len(src))
    v = test_emb_dev[src.to(device)]
    q, attn, g = fusion(v, cond_emb, sign_ids, None, return_attn=True)
    movement = 1 - (q * v).sum(dim=-1)
    return conds, attn.cpu().numpy(), movement.cpu().numpy()


composed = [e for e in annotations if len(parse_query(e["query"])) >= 2]
fig, axes = plt.subplots(1, len(composed), figsize=(3.2 * len(composed), 3.2), squeeze=False)
for ax, entry in zip(axes[0], composed):
    conds, attn, movement = attention_report(entry)
    labels = [("+" if s > 0 else "-") + a for a, s in conds]
    bars = ax.bar(range(len(conds)), attn.mean(axis=0), color="#2a78d6")
    ax.bar_label(bars, fmt="%.2f", padding=2, fontsize=8, color="#52514e")
    ax.set_xticks(range(len(conds)), labels, rotation=30, ha="right", fontsize=8)
    ax.set_ylim(0, 1)
    ax.grid(axis="y", alpha=0.25)
    ax.spines[["top", "right"]].set_visible(False)
    ax.set_title(entry["query"], fontsize=9)
    print(f"{entry['query']:<45} mean movement (1 - cos): {movement.mean():.3f}")
axes[0][0].set_ylabel("mean attention weight")
fig.suptitle("How the fusion module distributes attention across conditions")
plt.tight_layout()
plt.show()

## 9. Conclusions

> **TODO after running:** summarize — (i) how much each level improves macro R@K and why,
> (ii) what the attention/gate analysis reveals about how conditions are weighted,
> (iii) limitations (e.g. attributes rare in the corpus, correlated attributes) and
> possible extensions (visual directions as condition tokens, per-condition gates,
> larger CLIP backbones as an extra-model comparison).

### References

- Radford et al., *Learning Transferable Visual Models From Natural Language Supervision*, ICML 2021 (CLIP).
- Berasi et al., *Not Only Text: Exploring Compositionality of Visual Representations in VLMs*, CVPR 2025 (GDE; geodesic decomposition, log/exp maps).
- Lim et al., *CLAY: Conditional Visual Similarity Modulation in Vision-Language Embedding Space*, CVPR 2026 (frozen visual database, conditional similarity).
- Trager et al., *Linear Spaces of Meanings: Compositional Structures in VLMs*, ICCV 2023 (linear attribute directions).
- Yuksekgonul et al., *When and Why VLMs Behave like Bags-of-Words*, ICLR 2023 (compositionality limits, negation).
- Liu et al., *Deep Learning Face Attributes in the Wild*, ICCV 2015 (CelebA).
- Oord et al., *Representation Learning with Contrastive Predictive Coding*, 2018 (InfoNCE).

*All code in this notebook was written from scratch for this project, building only on the
course-provided skeleton (dataset setup, metric function).*